# 🏏 SupeRR Selector Season 2: Comprehensive Pressure Analytics
## Identifying the Most Reliable Players Under Pressure (IPL & T20I: 2022–2026)
**Rajasthan Royals Data Hackathon — Round 1**

---

### Executive Overview & Analytical Flow
This interactive analytics notebook guides you end-to-end through our data-driven player evaluation pipeline:
1. **Dataset Summary & Exploration**: Ingesting and auditing 200,000+ ball-by-ball records across IPL and T20 International cricket.
2. **Leak-Free State Tracking & Cleaning**: Rigorous pre-delivery state extraction using grouped `.shift(1)` operations to prevent lookahead bias.
3. **Asymmetric Pressure Modeling**: Formulating mathematically sound **Batting Pressure Index ($BPI$)** and **Bowling Pressure Index ($QPI$)** reflecting distinct situational psychology.
4. **Baseline vs. High-Pressure Comparison**: Quantifying player performance across Strike Rate, Boundary %, Dismissal Rate, Economy, Dot Ball %, and Wicket Strike Rate.
5. **The Calibrated Clutch Model**: Eliminating the "Elite Penalty" via ANCOVA Residual Lift, Empirical Bayes (James-Stein) Shrinkage, and Match-to-Match Consistency Weighting.
6. **Top 10 Batters & Bowlers Under Pressure**: Deep-dive statistical profiles, leaderboards, and tactical auction recommendations.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Set styling
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["figure.dpi"] = 120

# Add project root to sys.path
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pressure import clean, features, models, config as C

print("Setup completed successfully. Working directory:", ROOT)


---
## 1. Dataset Summary & Exploratory Data Analysis

We begin by loading the historical datasets provided for the competition:
- `ball_by_ball.csv`: Delivery-level records with run types, dismissals, and extras.
- `matches.csv`: Match-level metadata including venues, dates, toss decisions, and outcomes.
- `competitions.csv`: Tournament mappings and seasons (2022–2026).
- `teams.csv` & `players.parquet`: Entity definitions and roster mappings.


In [ ]:
# Load raw and interim tables
matches = pd.read_parquet("data/interim/matches.parquet")
players = pd.read_parquet("data/interim/players.parquet")
deliveries = pd.read_parquet("data/processed/deliveries_pressure.parquet")

print("==================================================")
print("📊 DATASET AT A GLANCE")
print("==================================================")
print(f"Total Matches Recorded:      {len(matches):,}")
print(f"Usable Clean Matches:        {matches['usable'].sum():,} ({matches['usable'].mean()*100:.1f}%)")
print(f"Total Deliveries Processed:  {len(deliveries):,}")
print(f"IPL Deliveries:              {(deliveries['league'] == 'IPL').sum():,}")
print(f"T20I Deliveries:             {(deliveries['league'] == 'T20I').sum():,}")
print(f"Unique Players Tracked:      {len(players):,}")

# Distribution of Matches by League and Season
match_dist = matches[matches['usable']].groupby(['season', 'league']).size().unstack(fill_value=0)
display(match_dist)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Matches per Season by League
match_dist.plot(kind="bar", stacked=True, ax=axes[0], color=["#d9534f", "#0275d8"])
axes[0].set_title("Usable Matches by Season & League", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Season")
axes[0].set_ylabel("Match Count")
axes[0].grid(axis="y", linestyle=":", alpha=0.6)

# 2. Deliveries per Match Phase
phase_counts = deliveries["phase"].value_counts()[["Powerplay", "Middle", "Death"]]
axes[1].pie(phase_counts, labels=phase_counts.index, autopct="%1.1f%%", colors=["#5bc0de", "#f0ad4e", "#d9534f"], startangle=140)
axes[1].set_title("Delivery Distribution by Match Phase", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()


---
## 2. Leak-Free Pre-Delivery State Tracking & Quality Control

A fundamental vulnerability in sports analytics is **look-ahead leakage**—using statistics that occur during or after a delivery to evaluate the pressure before the delivery.

Our pipeline implements strict pre-delivery state tracking:
* Chronologically sorts by `(match_id, innings, innings_seq)`.
* Computes `runs_before`, `wickets_before`, and `legal_balls_before` using `.shift(1)` within each innings.
* Pre-delivery collapse indicator (`wickets_last_18`) is strictly lagged to prevent current dismissal leakage.
* Balls remaining strictly bounded: $	ext{balls\_remaining} = \max(120 - 	ext{legal\_balls\_before}, 0)$.


In [ ]:
# Demonstrate leak-free state tracking on a sample chase
sample_match = deliveries[(deliveries['innings'] == 2) & (deliveries['balls_remaining'] <= 18)].head(8)
cols_to_view = ["match_id", "innings", "over0", "over_ball_num", "striker_name", "bowler_name", "runs_before", "wickets_lost", "balls_remaining", "runs_required", "rrr", "bpi", "qpi"]
display(sample_match[cols_to_view])


---
## 3. Asymmetric Match Pressure: BPI vs. QPI

Batters and bowlers experience different psychological stress on the same delivery:
* **Batting Pressure Index ($BPI$)**: Driven by escalating Required Run Rate (chase urgency), resource depletion (wickets down), and delivery scarcity.
* **Bowling Pressure Index ($QPI$)**: Driven by defensive margin volatility (close finishes), death-over slog threat, and opposition wickets in hand.

Both indices are bounded within $[0, 1]$ and anchored to empirical league-specific par pacing curves.


In [ ]:
plt.figure(figsize=(11, 4.8))
plt.hist(deliveries["bpi"], bins=50, density=True, alpha=0.45, color="#d9534f", label="Batting Pressure Index (BPI)")
plt.hist(deliveries["qpi"], bins=50, density=True, alpha=0.45, color="#0275d8", label="Bowling Pressure Index (QPI)")

tau_b = deliveries["bpi"].quantile(0.75)
tau_q = deliveries["qpi"].quantile(0.75)

plt.axvline(tau_b, color="#d9534f", linestyle="--", linewidth=1.8, label=f"Batting 75th %ile Threshold ({tau_b:.3f})")
plt.axvline(tau_q, color="#0275d8", linestyle="--", linewidth=1.8, label=f"Bowling 75th %ile Threshold ({tau_q:.3f})")

plt.title("Distribution of Continuous Match Pressure: BPI vs QPI", fontsize=13, fontweight="bold")
plt.xlabel("Pressure Index [0 - 1]")
plt.ylabel("Probability Density")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

print(f"High Pressure Cutoff: Batters >= {tau_b:.3f} | Bowlers >= {tau_q:.3f}")


---
## 4. The Calibrated Clutch Model: Solving the "Elite Penalty"

### The Fundamental Flaw of Naive Subtraction:
If we define Clutch as $\Delta = 	ext{Pressure Performance} - 	ext{Baseline Performance}$:
* An elite player like **Jasprit Bumrah** has an exceptional baseline economy of 6.4 RPO. Under extreme death pressure defending small totals, he concedes 7.6 RPO. His raw delta is **-1.2 RPO**, incorrectly labeling him as "un-clutch"!
* A below-average bowler conceding 11.5 RPO normally who concedes 11.0 RPO under pressure gets a **+0.5 RPO positive delta**, artificially inflating them!

### The Solution: ANCOVA Residual Lift Model
Performance regresses toward the mean under extreme pressure ($eta pprox 0.34$).
We estimate the regression line:
$$Z_{	ext{pressure}, i} = lpha + eta \cdot Z_{	ext{baseline}, i} + 	ext{Residual Lift}_i$$

* **Residual Lift**: How much better a player performed than expected *given their baseline level*.
* **Empirical Bayes (James-Stein) Shrinkage**: Pulls small-sample estimates toward the prior: $\lambda_i = rac{N_i}{N_i + K}$ with $K = 45.0$.
* **Match-to-Match Consistency Weighting**: Rewards players with high impact across multiple matches.


In [ ]:
# Run the calibrated evaluation pipeline
batters_ranked = models.evaluate_batters(deliveries, min_hp_balls=100, k_shrink=45.0)
bowlers_ranked = models.evaluate_bowlers(deliveries, min_hp_balls=100, k_shrink=45.0)

print(f"Qualified Batters Evaluated (>= 100 HP balls): {len(batters_ranked)}")
print(f"Qualified Bowlers Evaluated (>= 100 HP balls): {len(bowlers_ranked)}")


---
## 5. 🏆 Top 10 Batters Under Pressure

The 10 best-performing batters under pressure, ordered from Rank 1 to Rank 10.


In [ ]:
top10_bat = batters_ranked.head(10)[[
    "rank", "striker_name", "tot_hp_balls", "avg_hp_sr", "avg_norm_sr", 
    "avg_hp_bnd", "residual_lift", "impact_rate", "clutch_score"
]].copy()

top10_bat.columns = [
    "Rank", "Batter Name", "HP Balls", "HP Strike Rate", "Normal Strike Rate", 
    "HP Boundary %", "Residual Lift", "Impact Match %", "Clutch Score (CCS)"
]

top10_bat["HP Strike Rate"] = top10_bat["HP Strike Rate"].map("{:.1f}".format)
top10_bat["Normal Strike Rate"] = top10_bat["Normal Strike Rate"].map("{:.1f}".format)
top10_bat["HP Boundary %"] = (top10_bat["HP Boundary %"]).map("{:.1f}%".format)
top10_bat["Residual Lift"] = top10_bat["Residual Lift"].map("{:+.3f}".format)
top10_bat["Impact Match %"] = (top10_bat["Impact Match %"] * 100).map("{:.1f}%".format)
top10_bat["Clutch Score (CCS)"] = top10_bat["Clutch Score (CCS)"].map("{:.3f}".format)

display(top10_bat.style.hide(axis="index"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Bar plot of Top 10 Batters
top10_b_plot = batters_ranked.head(10).sort_values("clutch_score", ascending=True)
bars = axes[0].barh(top10_b_plot["striker_name"], top10_b_plot["clutch_score"], color="#d9534f")
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 0.02, bar.get_y() + bar.get_height()/2, f"{w:.3f}", va="center", fontsize=9, fontweight="bold")
axes[0].set_title("Top 10 Batters by Calibrated Clutch Score", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Clutch Score (CCS)")
axes[0].set_xlim(0, max(top10_b_plot["clutch_score"]) * 1.15)
axes[0].grid(axis="x", linestyle=":", alpha=0.6)

# 2. Normal SR vs Pressure SR Scatter
axes[1].scatter(batters_ranked["avg_norm_sr"], batters_ranked["avg_hp_sr"], c=batters_ranked["clutch_score"], cmap="Reds", s=batters_ranked["tot_hp_balls"]*0.5, alpha=0.7, edgecolors="grey")
lims = [110, 215]
axes[1].plot(lims, lims, "k--", alpha=0.5, label="Parity Line (No Elevation)")
for _, r in batters_ranked.head(10).iterrows():
    axes[1].annotate(r["striker_name"], (r["avg_norm_sr"]+1.2, r["avg_hp_sr"]+1.2), fontsize=8.5, fontweight="bold")
axes[1].set_title("Batters: Normal SR vs High-Pressure SR (Size = Volume)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Normal Baseline Strike Rate")
axes[1].set_ylabel("High-Pressure Strike Rate")
axes[1].legend()
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


---
## 6. 🏆 Top 10 Bowlers Under Pressure

The 10 best-performing bowlers under pressure, ordered from Rank 1 to Rank 10.


In [ ]:
top10_bowl = bowlers_ranked.head(10)[[
    "rank", "bowler_name", "tot_hp_balls", "avg_hp_econ", "avg_norm_econ", 
    "avg_hp_dot", "avg_hp_wkt", "residual_lift", "impact_rate", "clutch_score"
]].copy()

top10_bowl.columns = [
    "Rank", "Bowler Name", "HP Balls", "HP Economy", "Normal Economy", 
    "HP Dot %", "HP Wkt %", "Residual Lift", "Impact Match %", "Clutch Score (CCS)"
]

top10_bowl["HP Economy"] = top10_bowl["HP Economy"].map("{:.2f}".format)
top10_bowl["Normal Economy"] = top10_bowl["Normal Economy"].map("{:.2f}".format)
top10_bowl["HP Dot %"] = (top10_bowl["HP Dot %"]).map("{:.1f}%".format)
top10_bowl["HP Wkt %"] = (top10_bowl["HP Wkt %"]).map("{:.1f}%".format)
top10_bowl["Residual Lift"] = top10_bowl["Residual Lift"].map("{:+.3f}".format)
top10_bowl["Impact Match %"] = (top10_bowl["Impact Match %"] * 100).map("{:.1f}%".format)
top10_bowl["Clutch Score (CCS)"] = top10_bowl["Clutch Score (CCS)"].map("{:.3f}".format)

display(top10_bowl.style.hide(axis="index"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Bar plot of Top 10 Bowlers
top10_w_plot = bowlers_ranked.head(10).sort_values("clutch_score", ascending=True)
bars = axes[0].barh(top10_w_plot["bowler_name"], top10_w_plot["clutch_score"], color="#0275d8")
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 0.012, bar.get_y() + bar.get_height()/2, f"{w:.3f}", va="center", fontsize=9, fontweight="bold")
axes[0].set_title("Top 10 Bowlers by Calibrated Clutch Score", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Clutch Score (CCS)")
axes[0].set_xlim(0, max(top10_w_plot["clutch_score"]) * 1.15)
axes[0].grid(axis="x", linestyle=":", alpha=0.6)

# 2. Normal Economy vs Pressure Economy Scatter
axes[1].scatter(bowlers_ranked["avg_norm_econ"], bowlers_ranked["avg_hp_econ"], c=bowlers_ranked["clutch_score"], cmap="Blues", s=bowlers_ranked["tot_hp_balls"]*0.4, alpha=0.7, edgecolors="grey")
lims = [5.5, 11.5]
axes[1].plot(lims, lims, "k--", alpha=0.5, label="Parity Line (No Elevation)")
for _, r in bowlers_ranked.head(10).iterrows():
    axes[1].annotate(r["bowler_name"], (r["avg_norm_econ"]+0.1, r["avg_hp_econ"]+0.1), fontsize=8.5, fontweight="bold")
axes[1].set_title("Bowlers: Normal Economy vs High-Pressure Economy (Size = Volume)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Normal Baseline Economy")
axes[1].set_ylabel("High-Pressure Economy (Lower is Better)")
axes[1].legend()
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


---
## 7. Strategic Auction Recommendations for Rajasthan Royals

### Batting Unit Insights:
1. **Jos Buttler & Suryakumar Yadav**: Elite dual-threat assets. Buttler provides explosive power in pressure chases (HP SR: 199.4), while Suryakumar provides unmatched consistency (78.9% impact rate over 475 pressure deliveries).
2. **Naman Dhir**: Premium domestic emerging finisher. With an astonishing 206.0 HP strike rate and 30% boundary rate across 150 pressure balls, he represents an extraordinary high-upside acquisition.
3. **Nicholas Pooran & Glenn Maxwell**: Indispensable middle-order game changers capable of flipping impossible run chases.

### Bowling Unit Insights:
1. **Jasprit Bumrah**: The premier death bowler in world cricket. Bowling 552 pressure deliveries at 7.61 RPO (2.5+ runs below league average) with a 75.5% match impact rate.
2. **Noor Ahmad & Rashid Khan**: Mystery spin is the ultimate pressure dampener. Noor Ahmad's 40.2% dot-ball rate and 7.8% wicket rate make him the top-ranked pressure spinner.
3. **Ben Dwarshuis & Taskin Ahmed**: High-value pace options who elevate their wicket-taking strike rate under stress.

---
### Artifacts & Deliverables Exported:
* `outputs/top_10_batters.csv` & `top_10_batters.json`
* `outputs/top_10_bowlers.csv` & `top_10_bowlers.json`
* `outputs/Methodology_Report.md` (Competition-ready document upload)
* `data/processed/deliveries_pressure.parquet` (Full enriched ball-by-ball dataset)
